# **Analyse rapide de methadonn?es**

L'objectif de ce notbook est de faire une breve analyse de meth?donn?e disponibles sur l'ensemble des images de radiographie de torse. 

<html>
<head>
    <title>Todo List</title>
    <style>
        .todo {
            color: rgb(250, 154, 64);
        }
    </style>
</head>

<body>
    <h1>Todo List</h1>
    <ul>
        <li class="todo">Introduction</li>
        <li class="todo">Conclusion</li>
        <li class="todo">New Data for pre-processing of images</li>
    </ul>
</body>

</html>


In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', append=True, category=UserWarning)
from aif360.datasets import StandardDataset
from aif360.algorithms.preprocessing import Reweighing
from sklearn.preprocessing import LabelEncoder

In [ ]:
df = pd.read_csv('clem/metadata.csv')

<p style="color:rgb(250, 154, 64);">
      On regarde qq staistique sur les jeux de donn?es </p>

# Analyse des M?tadonn?es et Pr?traitement par Reweighting

Dans ce notebook, nous allons analyser les m?tadonn?es d'un jeu de donn?es m?dical o? :
- Les **colonnes sensibles** sont `Patient Age` et `Patient Gender`.
- La **variable cible** est `Finding Labels`.  
  Si la valeur est `"No Finding"`, cela signifie qu'il n'y a **aucune maladie** d?tect?e.  
  Dans le cas contraire, la cha?ne contient une ou plusieurs maladies (s?par?es par `|`).

L'objectif est de :

1. Pr?parer les donn?es en cr?ant une variable binaire `disease` (0 pour *No Finding*, 1 pour pr?sence de maladie).
2. R?aliser une analyse univari?e et bivari?e en utilisant Plotly Express pour visualiser les distributions et relations.
3. Calculer des poids pour chaque ?chantillon via un **reweighting** (pour compenser un ?ventuel d?s?quilibre de classes) et stocker ces poids dans la colonne `WEIGHTS` (d?j? initialis?s ? 1.0).

"""


In [ ]:
# Affichage d'un aper?u des donn?es
print("Aper?u des premi?res lignes:")
display(df.head())

# V?rifier la pr?sence des colonnes sensibles et de la colonne cible
print("Colonnes du DataFrame:", df.columns.tolist())

## 1. Pr?paration des donn?es

In [ ]:
# Cr?ation de la variable cible binaire
#  - Si Finding Labels == "No Finding" => 0 (aucune maladie)
#  - Sinon => 1 (pr?sence d'une ou plusieurs maladies)
df['disease'] = df['Finding Labels'].apply(lambda x: 0 if x == "No Finding" else 1)

In [ ]:
# Correction des ?ges incorrects : remplacement des ?ges > 140 par la m?diane des ?ges valides
valid_age_mask = df['Patient Age'] <= 140
median_age = df.loc[valid_age_mask, 'Patient Age'].median()
df.loc[~valid_age_mask, 'Patient Age'] = median_age

In [ ]:
# Calcul de la distribution des malades/sains
distribution = df['disease'].value_counts().reset_index()
distribution.columns = ['disease', 'count']
distribution['disease'] = distribution['disease'].map({0: 'Sain', 1: 'Malade'})

# Affichage de la distribution avec un pie chart
fig_pie = px.pie(distribution, values='count', names='disease', 
                 title="R?partition des malades/sains parmi le jeu de donn?es",
                 color_discrete_sequence=px.colors.qualitative.Pastel)
fig_pie.show()

## 2. Analyse

### 2.1 Analyse Univari?e

In [ ]:
# 1. Distribution de Patient Age
fig_age = px.histogram(df, 
                       x='Patient Age', 
                       title="Distribution de l'?ge des patients",
                       nbins=40,
                       color_discrete_sequence=['skyblue'],
                       opacity=0.75,
                       labels={'Patient Age': '?ge du patient'},
                       )
fig_age.show()

In [ ]:
#2 Distribution du Genre

gender_distribution = df['Patient Gender'].value_counts().reset_index()
gender_distribution.columns = ['Patient Gender', 'count']

# Affichage de la distribution avec un pie chart
fig_pie_gender = px.pie(gender_distribution, values='count', names='Patient Gender',
                        title="R?partition du Genre ",
                        color_discrete_sequence=px.colors.qualitative.Pastel)
fig_pie_gender.show()

### 2.2 Analyse bivari?e

In [ ]:
# 1. Boxplot de Patient Age en fonction de la variable cible
fig_box_age = px.box(df, x='disease', y='Patient Age', 
                     title="?ge des patients selon la pr?sence ou l'absence de maladie",
                     labels={'disease': 'Pr?sence de maladie (0 = Sain, 1 = Malade)', 'Patient Age': '?ge du patient'})
fig_box_age.show()


In [ ]:
# Calcul du taux de malades par genre
df_gender_disease = df.groupby('Patient Gender')['disease'].value_counts(normalize=True).unstack().reset_index()

# Graphique ? barres comparant le taux de malades chez les hommes et les femmes
fig_bar = px.bar(df_gender_disease, 
                 x='Patient Gender', 
                 y=[0, 1],  # 0 correspond aux sains, 1 correspond aux malades
                 labels={'Patient Gender': 'Genre', 'value': 'Taux de Malade'}, 
                 title="Comparaison du Taux de Malades entre Hommes et Femmes",
                 barmode='group',  
                 text_auto=True)  

fig_bar.show()

On remarque qu'il y a une l?g?re diff?rence de r?partition des non-malades entre femmes et hommes .

## Repond?ration

In [ ]:
dfr = df.copy()
label_encoder = LabelEncoder()
for column in dfr.select_dtypes(include=['object']).columns:
    dfr[column] = label_encoder.fit_transform(dfr[column])

### Par le genre

In [ ]:
dataset_gender = StandardDataset(
    dfr,
    label_name="disease",              # Variable ? pr?dire
    favorable_classes=[0],             # 0 = absence de maladie (classe favorable)
    protected_attribute_names=["Patient Gender"],
    privileged_classes=[[1]]           # Groupe privil?gi? : patients avec Patient Gender == 1
)
reweigher_gender = Reweighing(
    unprivileged_groups=[{'Patient Gender': 0}],  # Non privil?gi?
    privileged_groups=[{'Patient Gender': 1}]       # Privil?gi?
)
dataset_transf_gender = reweigher_gender.fit_transform(dataset_gender)
dfr['WEIGHTS_GENDER'] = dataset_transf_gender.instance_weights

### Par l'?ge (selon les quartiles)

In [ ]:
# --- 3. Reweighting par l'?ge ---
# On d?finit une variable binaire "Age Privileged" en fonction du label 'disease' et des intervalles d'?ge :
#   - Pour les patients sans maladie (disease == 0) : 34 <= Patient Age <= 59 est privil?gi?
#   - Pour les patients malades (disease == 1) : 42 <= Patient Age <= 62 est privil?gi?
def age_privileged(row):
    if row['disease'] == 0:
        return int(34 <= row['Patient Age'] <= 59)
    else:
        return int(42 <= row['Patient Age'] <= 62)

dfr['Age Privileged'] = dfr.apply(age_privileged, axis=1)

dataset_age = StandardDataset(
    dfr,
    label_name="disease",              # Variable ? pr?dire
    favorable_classes=[0],
    protected_attribute_names=["Age Privileged"],
    privileged_classes=[[1]]           # Groupe privil?gi? : ?ge dans l'intervalle d?fini
)
reweigher_age = Reweighing(
    unprivileged_groups=[{'Age Privileged': 0}],
    privileged_groups=[{'Age Privileged': 1}]
)
dataset_transf_age = reweigher_age.fit_transform(dataset_age)
dfr['WEIGHTS_AGE'] = dataset_transf_age.instance_weights

### Par les deux

In [ ]:
# On d?finit "Combined Privileged" comme 1 si le patient a ? la fois Patient Gender == 1 et Age Privileged == 1, sinon 0
dfr['Combined Privileged'] = ((dfr['Patient Gender'] == 1) & (dfr['Age Privileged'] == 1)).astype(int)

dataset_combined = StandardDataset(
    dfr,
    label_name="disease",              # Variable ? pr?dire
    favorable_classes=[0],
    protected_attribute_names=["Combined Privileged"],
    privileged_classes=[[1]]           # Groupe privil?gi? si les deux conditions sont remplies
)
reweigher_combined = Reweighing(
    unprivileged_groups=[{'Combined Privileged': 0}],
    privileged_groups=[{'Combined Privileged': 1}]
)
dataset_transf_combined = reweigher_combined.fit_transform(dataset_combined)
dfr['WEIGHTS_COMBINED'] = dataset_transf_combined.instance_weights

# --- V?rification rapide : affichage des 10 premi?res lignes avec les colonnes de poids ---
print(dfr[['Patient Gender', 'Patient Age', 'disease', 'Age Privileged', 'Combined Privileged',
           'WEIGHTS_GENDER', 'WEIGHTS_AGE', 'WEIGHTS_COMBINED']].head(10))

In [ ]:
df["WEIGHTS"] = dfr["WEIGHTS"]

In [ ]:
df0 = pd.read_csv('DATASET/metadata.csv')
df1 = df0.copy()
df2 = df1.copy()

In [ ]:
df0['WEIGHTS']=dfr['WEIGHTS_GENDER'].values

In [ ]:
df1['WEIGHTS']=dfr['WEIGHTS_AGE'].values

In [ ]:
df2['WEIGHTS']=dfr['WEIGHTS_COMBINED'].values

### Faire les csv pour le training

In [ ]:
df0.to_csv("./1_clem/metadata_gender.csv",index=False)
df1.to_csv("./1_clem/metadata_age.csv",index=False)
df2.to_csv("./1_clem/metadata_both.csv",index=False)


In [ ]:
df0.to_csv("./clem/metadata_gender.csv",index=False)
df1.to_csv("./clem/metadata_age.csv",index=False)
df2.to_csv("./clem/metadata_both.csv",index=False)

In [ ]:
test = pd.read_csv("1clem_out_age/preds.csv")
test['preds']